# LNCLIP-DF — Final Training Notebook

**Key features:**
- Reads directly from Drive (no local copy — avoids corruption)
- Resumes training from checkpoint if it exists
- All imports inside each cell (no NameError on re-runs)
- Class weighting + LN-tuning + discriminative LR
- Early stopping patience=5

**Runtime:** ~2.5h preprocessing (first run) + ~4h training from Drive

**For rerun:** Skip Cell 4 (preprocessing already cached). Start from Cell 1.

In [ ]:
# ═══ CELL 1: Install ══════════════════════════════════════════
import torch
assert torch.cuda.is_available(), 'Runtime → Change runtime type → T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)}')
!pip install -q open_clip_torch insightface onnxruntime-gpu opencv-python-headless scipy scikit-learn tqdm pandas matplotlib
print('✓ Done')

In [ ]:
# ═══ CELL 2: Mount + Clone ════════════════════════════════════
from google.colab import drive
import os, sys, glob
from pathlib import Path
import pandas as pd

drive.mount('/content/drive')

if not os.path.exists('/content/eidon-deepfake'):
    !git clone https://github.com/rxbinsingh/eidon-deepfake /content/eidon-deepfake
else:
    !git -C /content/eidon-deepfake pull --quiet

os.chdir('/content/eidon-deepfake')
sys.path.insert(0, '/content/eidon-deepfake')

BASE      = '/content/drive/MyDrive/VIPER'
DATA_DIRS = [f'{BASE}/dataset_production', f'{BASE}/dataset_week2', f'{BASE}/dfd_dataset']
CACHE_DIR = f'{BASE}/cache_dual'
SAVE_DIR  = f'{BASE}/checkpoints_final'
os.makedirs(CACHE_DIR, exist_ok=True)
os.makedirs(SAVE_DIR, exist_ok=True)

total = 0
for d in DATA_DIRS:
    if os.path.exists(f'{d}/metadata.csv'):
        n = len(pd.read_csv(f'{d}/metadata.csv'))
        total += n
        print(f'  ✓ {Path(d).name}: {n} videos')
    else:
        print(f'  ✗ {Path(d).name}: NOT FOUND')
print(f'Total: {total}')
print(f'Cached: {len(glob.glob(CACHE_DIR+"/*.npz"))} | Save: {SAVE_DIR}')

In [ ]:
# ═══ CELL 3: Find all videos ══════════════════════════════════
import numpy as np
from pathlib import Path
import pandas as pd, os, random

def get_all_videos():
    samples = []
    for d in DATA_DIRS:
        meta = f'{d}/metadata.csv'
        if not os.path.exists(meta): continue
        df = pd.read_csv(meta)
        found = 0
        for _, row in df.iterrows():
            label_str = row.get('label', 'unknown')
            label = 0 if label_str == 'real' else 1
            cat = row.get('category', label_str)
            fname = row['filename']
            paths = [f'{d}/{cat}/{fname}', f'{d}/{label_str}/{fname}']
            if 'original_path' in row and pd.notna(row['original_path']):
                paths += [f'{d}/{row["original_path"]}',
                          f'{d}/{Path(row["original_path"]).parent}/{fname}']
            if 'dfd' in d.lower():
                paths += [f'{d}/DFD_original sequences/{fname}',
                          f'{d}/DFD_manipulated_sequences/{fname}',
                          f'{d}/DFD_manipulated_sequences/DFD_manipulated_sequences/{fname}']
            for vp in paths:
                if os.path.exists(vp):
                    samples.append((vp, label, cat)); found += 1; break
        print(f'  {Path(d).name}: {found}')
    return samples

all_videos = get_all_videos()
print(f'Total: {len(all_videos)} | Real: {sum(1 for _,l,_ in all_videos if l==0)} | Fake: {sum(1 for _,l,_ in all_videos if l==1)}')

In [ ]:
# ═══ CELL 4: Preprocessing (SKIP if already cached) ══════════
# Run only on first-time setup. Check count first.
import cv2, numpy as np, os, glob
from tqdm import tqdm
from pathlib import Path
from src.preprocessing import preprocess_video
try:
    import src.anchor_extractor as ae
    ae.build_biomech_anchor = lambda frames: None
except: pass

already = len(glob.glob(f'{CACHE_DIR}/*.npz'))
print(f'Already cached: {already}/{len(all_videos)}')
if already >= len(all_videos) * 0.9:
    print('Cache is >90% complete — skip this cell and go to Cell 5')
else:
    ok, fail = 0, 0
    for vp, label, cat in tqdm(all_videos):
        cp = f'{CACHE_DIR}/{Path(vp).stem}.npz'
        if os.path.exists(cp): ok += 1; continue
        try:
            p = preprocess_video(vp, num_frames=16, n_anchor=8)
            if not p['valid']: fail += 1; continue
            full = [cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in p['full_frames']]
            sd = {'full_frames': np.stack(full), 'label': np.array(label), 'face_valid': np.array(p['face_valid'])}
            if p['face_valid'] and p['face_crops']:
                sd['face_crops'] = np.stack([cv2.cvtColor(c, cv2.COLOR_BGR2RGB) for c in p['face_crops']])
            np.savez_compressed(cp, **sd)
            ok += 1
        except: fail += 1
    print(f'Done. OK={ok} Fail={fail}')

In [ ]:
# ═══ CELL 5: Build Model + Datasets (reads from Drive) ═══════
import cv2, numpy as np, os, random, glob, time
from pathlib import Path
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torchvision import transforms as T
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
from PIL import Image
import open_clip
from src.lnclip_model import LNCLIPDeepfakeDetector, ArcMarginProduct

device = 'cuda'
random.seed(42); np.random.seed(42); torch.manual_seed(42)

CLIP_TF = T.Compose([T.Resize(224), T.CenterCrop(224), T.ToTensor(),
    T.Normalize([0.48145466,0.4578275,0.40821073],[0.26862954,0.26130258,0.27577711])])

class CompressAug:
    def __init__(self, p=0.4):
        self.p = p
    def __call__(self, img):
        import cv2, numpy as np
        if random.random() > self.p: return img
        arr = np.array(img)
        if random.random() < 0.6:
            q = random.randint(40, 95)
            _, enc = cv2.imencode('.jpg', cv2.cvtColor(arr, cv2.COLOR_RGB2BGR), [cv2.IMWRITE_JPEG_QUALITY, q])
            arr = cv2.cvtColor(cv2.imdecode(enc, cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        if random.random() < 0.3:
            s = random.uniform(0.5, 1.5); k = int(s*4)|1
            arr = cv2.GaussianBlur(arr, (k,k), s)
        return Image.fromarray(arr)

class DualDataset(Dataset):
    def __init__(self, samples, cache_dir, train=True):
        self.samples = [(p,l) for p,l,_ in samples if os.path.exists(f'{cache_dir}/{Path(p).stem}.npz')]
        self.cache_dir = cache_dir
        self.aug = CompressAug(0.4) if train else None
        self.flip = T.RandomHorizontalFlip(0.5) if train else T.RandomHorizontalFlip(0)
        print(f'  {"Train" if train else "Eval"}: {len(self.samples)}')
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        vp, label = self.samples[idx]
        cp = f'{self.cache_dir}/{Path(vp).stem}.npz'
        if not os.path.exists(cp):
            d = torch.zeros(16,3,224,224)
            return {'face':d,'full':d,'label':torch.tensor(label,dtype=torch.long)}
        try:
            data = np.load(cp)
            full = data['full_frames']
            fv = bool(data['face_valid'])
            fc = data.get('face_crops', None)
        except:
            d = torch.zeros(16,3,224,224)
            return {'face':d,'full':d,'label':torch.tensor(label,dtype=torch.long)}
        def proc(arr, n=16):
            T_ = min(len(arr), n); ts = []
            for i in range(T_):
                img = Image.fromarray(arr[i])
                if self.aug: img = self.aug(img)
                img = self.flip(img)
                ts.append(CLIP_TF(img))
            while len(ts)<n: ts.append(ts[-1])
            return torch.stack(ts[:n])
        full_t = proc(full)
        face_t = proc(fc) if (fv and fc is not None and len(fc)>=4) else torch.zeros_like(full_t)
        return {'face':face_t,'full':full_t,'label':torch.tensor(label,dtype=torch.long)}

# Split — fixed seed for reproducibility
shuffled = all_videos.copy(); random.seed(42); random.shuffle(shuffled)
n = len(shuffled)
train_s = shuffled[:int(.7*n)]; val_s = shuffled[int(.7*n):int(.8*n)]; test_s = shuffled[int(.8*n):]

# Read from Drive (reliable, no corruption risk)
train_ds = DualDataset(train_s, CACHE_DIR, train=True)
val_ds   = DualDataset(val_s,   CACHE_DIR, train=False)
test_ds  = DualDataset(test_s,  CACHE_DIR, train=False)
train_loader = DataLoader(train_ds, batch_size=8, shuffle=True,  num_workers=0, pin_memory=False)
val_loader   = DataLoader(val_ds,   batch_size=8, shuffle=False, num_workers=0, pin_memory=False)
test_loader  = DataLoader(test_ds,  batch_size=8, shuffle=False, num_workers=0, pin_memory=False)

# Build CLIP model
clip_model, _, _ = open_clip.create_model_and_transforms('ViT-L-14', pretrained='openai')
clip_model = clip_model.to(device).eval()

model = LNCLIPDeepfakeDetector(clip_model, num_trainable_layers=6).to(device)
model.classifier = nn.Linear(model.combined_dim, 2).to(device)
nn.init.xavier_uniform_(model.classifier.weight)
nn.init.zeros_(model.classifier.bias)
model.sphere_aug.noise_std = 0.0

def simple_forward(face, full, labels=None):
    emb = model.encode_frames(face, full)
    return model.classifier(emb), emb
model.forward = simple_forward

# Load checkpoint if exists (resume training)
CKPT = f'{SAVE_DIR}/best.pt'
start_epoch = 1
best_auc = 0.0
if os.path.exists(CKPT):
    model.load_state_dict(torch.load(CKPT, map_location=device))
    print(f'✓ Loaded checkpoint from {CKPT}')
    print('  Evaluating loaded model on val set...')
    model.eval()
    all_l, all_p = [], []
    with torch.no_grad():
        for b in val_loader:
            with torch.amp.autocast('cuda'):
                logits, _ = model(b['face'].to(device), b['full'].to(device))
            all_l.extend(b['label'].numpy())
            all_p.extend(F.softmax(logits,dim=-1)[:,1].cpu().numpy())
    best_auc = roc_auc_score(all_l, all_p) if len(set(all_l))>1 else 0
    print(f'  Checkpoint Val AUC: {best_auc:.4f}')
else:
    print('No checkpoint found. Training from scratch.')

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Trainable params: {trainable:,}')

In [ ]:
# ═══ CELL 6: Training with All Improvements ══════════════════
import torch, torch.nn as nn, torch.nn.functional as F
import time, json
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS   = 20
PATIENCE = 5

# Class-weighted loss
n_real_tr = sum(1 for _,l,_ in train_s if l==0)
n_fake_tr = sum(1 for _,l,_ in train_s if l==1)
class_weights = torch.tensor([n_fake_tr/n_real_tr, 1.0]).to(device)
print(f'Class weights: real={class_weights[0]:.2f}, fake={class_weights[1]:.2f}')

# Discriminative LR: LN slower, classifier faster
ln_params   = [p for n_,p in model.named_parameters() if p.requires_grad and 'classifier' not in n_]
head_params = list(model.classifier.parameters())
optimizer   = AdamW([{'params':ln_params,'lr':1e-4},{'params':head_params,'lr':3e-4}], weight_decay=1e-4)
scheduler   = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)
scaler      = torch.amp.GradScaler('cuda')

history     = []
patience_ct = 0
t_start     = time.time()

def run_train(epoch):
    model.train()
    model.sphere_aug.noise_std = 0.03 if epoch > 5 else 0.0
    all_l, all_p, loss_sum = [], [], 0.0
    for b in tqdm(train_loader, leave=False):
        face, full, labels = b['face'].to(device), b['full'].to(device), b['label'].to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits, emb = model(face, full, labels)
            loss = F.cross_entropy(logits, labels, weight=class_weights)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer); scaler.update()
        loss_sum += loss.item()
        all_l.extend(labels.cpu().numpy())
        all_p.extend(F.softmax(logits.detach(),dim=-1)[:,1].cpu().numpy())
    return loss_sum/len(train_loader), roc_auc_score(all_l,all_p) if len(set(all_l))>1 else 0

@torch.no_grad()
def run_eval(loader):
    model.eval()
    all_l, all_p = [], []
    for b in loader:
        face, full = b['face'].to(device), b['full'].to(device)
        with torch.amp.autocast('cuda'):
            logits, _ = model(face, full)
        all_l.extend(b['label'].numpy())
        all_p.extend(F.softmax(logits,dim=-1)[:,1].cpu().numpy())
    auc = roc_auc_score(all_l,all_p) if len(set(all_l))>1 else 0
    return auc, accuracy_score(all_l,[1 if p>.5 else 0 for p in all_p]), all_l, all_p

print(f'Training (max {EPOCHS} epochs, patience={PATIENCE})...\n')
for epoch in range(start_epoch, EPOCHS+1):
    t0 = time.time()
    tr_loss, tr_auc = run_train(epoch)
    vl_auc, vl_acc, _, _ = run_eval(val_loader)
    scheduler.step()
    elapsed = time.time() - t0
    total_min = (time.time()-t_start)/60
    phase = 'CE+noise' if epoch > 5 else 'CE'
    flag = ''
    if vl_auc > best_auc:
        best_auc = vl_auc; patience_ct = 0
        torch.save(model.state_dict(), f'{SAVE_DIR}/best.pt')
        flag = '  ← best'
    else:
        patience_ct += 1
    print(f'Ep {epoch:02d} [{phase}] Loss={tr_loss:.4f} Train={tr_auc:.4f} Val={vl_auc:.4f} Acc={vl_acc:.3f} ({elapsed:.0f}s|{total_min:.0f}min){flag}')
    history.append({'epoch':epoch,'tr_loss':tr_loss,'tr_auc':tr_auc,'vl_auc':vl_auc,'vl_acc':vl_acc})
    if patience_ct >= PATIENCE:
        print(f'Early stop at epoch {epoch}'); break

torch.save(model.state_dict(), f'{SAVE_DIR}/final.pt')
with open(f'{SAVE_DIR}/history.json','w') as f: json.dump(history,f,indent=2)
print(f'\nBest Val AUC: {best_auc:.4f} | Time: {(time.time()-t_start)/60:.0f} min')

In [ ]:
# ═══ CELL 7: Test Evaluation + TTA + Per-Category ════════════
import torch, torch.nn.functional as F
import numpy as np, json, os
from pathlib import Path
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt

model.load_state_dict(torch.load(f'{SAVE_DIR}/best.pt', map_location=device))
model.eval()

all_labels, all_probs = [], []
with torch.no_grad():
    for b in tqdm(test_loader, desc='Test+TTA'):
        face, full = b['face'].to(device), b['full'].to(device)
        with torch.amp.autocast('cuda'):
            l1,_ = model(face, full)
            l2,_ = model(torch.flip(face,[-1]), torch.flip(full,[-1]))
        probs = (F.softmax(l1,dim=-1)[:,1] + F.softmax(l2,dim=-1)[:,1]) / 2
        all_labels.extend(b['label'].numpy())
        all_probs.extend(probs.cpu().numpy())

auc = roc_auc_score(all_labels, all_probs)
preds = [1 if p>.5 else 0 for p in all_probs]
acc   = accuracy_score(all_labels, preds)
cm    = confusion_matrix(all_labels, preds)

print(f'\n{"═"*52}')
print(f'  FINAL TEST RESULTS')
print(f'{"═"*52}')
print(f'  AUC:      {auc:.4f}  (baseline: 0.9535)')
print(f'  Accuracy: {acc:.4f} ({acc*100:.1f}%)')
print(f'  TN={cm[0,0]} FP={cm[0,1]} | FN={cm[1,0]} TP={cm[1,1]}')
print(f'\n{classification_report(all_labels,preds,target_names=["Real","Fake"])}')

# Per-category
cat_l, cat_p = {}, {}
for (vp,label,cat),prob in zip(test_s,all_probs[:len(test_s)]):
    if cat not in cat_l: cat_l[cat],cat_p[cat]=[],[]
    cat_l[cat].append(label); cat_p[cat].append(prob)

print(f'\n{"Category":<28} {"AUC":>8} {"N":>5}')
print('─'*45)
rl,rp = cat_l.get('real',[]), cat_p.get('real',[])
per_type = {}
for cat in sorted(cat_l.keys()):
    if cat=='real': continue
    cl = rl+cat_l[cat]; cp_ = rp+cat_p[cat]
    if len(set(cl))<2: continue
    a = roc_auc_score(cl,cp_)
    per_type[cat] = round(a,4)
    print(f'{cat:<28} {a:>8.4f} {len(cat_l[cat]):>5}')
print('─'*45)
print(f'{"ALL":<28} {auc:>8.4f} {len(all_labels):>5}')

# Training curves
if os.path.exists(f'{SAVE_DIR}/history.json'):
    with open(f'{SAVE_DIR}/history.json') as f:
        history = json.load(f)
    ep = [h['epoch'] for h in history]
    fig,axes = plt.subplots(1,2,figsize=(12,4))
    axes[0].plot(ep,[h['tr_loss'] for h in history],'b-o',label='Train')
    axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True,alpha=0.3)
    axes[1].plot(ep,[h['tr_auc'] for h in history],'b-o',label='Train')
    axes[1].plot(ep,[h['vl_auc'] for h in history],'r-o',label='Val')
    axes[1].axhline(best_auc,color='green',linestyle='--',label=f'Best={best_auc:.4f}')
    axes[1].set_title('AUC'); axes[1].set_ylim(0.5,1.0); axes[1].legend(); axes[1].grid(True,alpha=0.3)
    plt.suptitle(f'Test AUC: {auc:.4f} | Δ from baseline: {auc-0.9535:+.4f}', fontweight='bold')
    plt.tight_layout()
    plt.savefig(f'{SAVE_DIR}/curves.png',dpi=150,bbox_inches='tight')
    plt.show()

# Save results
results = {'test_auc':round(auc,4),'accuracy':round(acc,4),'best_val_auc':round(best_auc,4),
           'baseline_auc':0.9535,'delta':round(auc-0.9535,4),'per_type_auc':per_type,'cm':cm.tolist()}
with open(f'{SAVE_DIR}/results_final.json','w') as f: json.dump(results,f,indent=2)
print(f'\nSaved. Δ from baseline: {auc-0.9535:+.4f}')